# Jigsaw: Topluluk kuralı ihlali tahmini

Bir Reddit yorumunun verilen kuralı ihlal etme olasılığını tahmin ediyoruz.
Yarışma: https://www.kaggle.com/competitions/jigsaw-agile-community-rules

Bütün model kodu bu notebook içinde. Yerel Python kurulumu, src klasörü veya proje ZIP'i gerekmiyor.
Hücreleri sırayla çalıştırın. Veri yükleme hücresinde train.csv, test.csv ve sample_submission.csv dosyalarını birlikte seçin.
CPU yeterli. Sonuçları son hücreden indirin; notebook'u Drive'a kaydedin.


## 1. Kütüphaneler


In [ ]:
%pip install -q "pandas>=2.0,<4" "scikit-learn>=1.3,<2"


## 2. Verileri yükle
Bu hücre her yeni Colab oturumunda yeniden çalıştırılır.


In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
required = {"train.csv", "test.csv", "sample_submission.csv"}
missing = required - uploaded.keys()
if missing:
    raise ValueError(f"Eksik dosyalar: {sorted(missing)}. Üç CSV'yi birlikte seçin.")
Path("data").mkdir(exist_ok=True)
for name in sorted(required):
    Path("data", name).write_bytes(uploaded[name])
del uploaded


Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


## 3. Veri okuma ve doğrulama ayrımı
Aynı yorumun eğitim ve doğrulamada bulunmasını engelliyoruz.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


def load_data(data_dir="data"):
    """CSV dosyalarını oku; test verisinde hedef sütunu bulunmaz."""
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    return train, test


def split_data(train, seed=42):
    """Aynı yorumun eğitim ve doğrulamaya birlikte girmesini engelle."""
    groups = train["body"].fillna("").str.lower().str.strip()
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    return train.iloc[train_idx].copy(), train.iloc[valid_idx].copy()


def make_text(frame):
    """Kural ve yorumu tek bir model girdisine dönüştür."""
    return "RULE: " + frame["rule"].fillna("") + " COMMENT: " + frame["body"].fillna("")


## 4. Modeller
Kelime ve karakter TF-IDF modellerini aynı ayrımlarda deniyoruz. Dummy yalnızca referans.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def build_models():
    return {
        "dummy": DummyClassifier(strategy="prior"),
        "tfidf_logistic": Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]),
        "char_logistic": Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                      min_df=2, max_features=30000)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]),
    }


## 5. Değerlendirme
AUC sıralama başarısını, accuracy doğru tahmin oranını gösterir. Accuracy ve F1 için eşik 0.5.


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


def print_table(title, frame):
    """DataFrame'i ek bağımlılık gerektirmeden çerçeveli tablo olarak yazdır."""
    formatted = frame.to_string(index=False, float_format=lambda value: f"{value:.4f}",
                                na_rep="N/A")
    lines = formatted.splitlines()
    width = max(len(line) for line in lines)
    border = "+-" + "-" * width + "-+"
    print(f"\n{title}")
    print(border)
    for index, line in enumerate(lines):
        print(f"| {line.ljust(width)} |")
        if index == 0:
            print(border)
    print(border)


def evaluate_model(frame, probabilities):
    """Genel ve kural bazında yerel doğrulama metriklerini hesapla."""
    target = frame["rule_violation"]
    predictions = (probabilities >= 0.5).astype(int)
    scored = frame[["rule", "rule_violation"]].copy()
    scored["probability"] = probabilities
    rule_scores = []
    for rule, group in scored.groupby("rule"):
        auc = (roc_auc_score(group["rule_violation"], group["probability"])
               if group["rule_violation"].nunique() == 2 else float("nan"))
        rule_scores.append({"rule": rule, "roc_auc": auc, "count": len(group)})
    per_rule = pd.DataFrame(rule_scores)
    metrics = {
        "roc_auc": roc_auc_score(target, probabilities),
        "mean_rule_auc": per_rule["roc_auc"].mean(),
        "accuracy": accuracy_score(target, predictions),
        "f1": f1_score(target, predictions, zero_division=0),
    }
    return metrics, per_rule


## 6. Eğit ve karşılaştır

7, 21 ve 42 tohumlarında yaklaşık %20 doğrulama ayırıyoruz. TF-IDF her ayrımın yalnızca eğitim bölümünde öğreniliyor.
Bu ayrımlar örtüşebilir. Sonuçlar resmi Kaggle skoru veya bağımsız test sonucu değil.


In [ ]:
root = Path.cwd()
seeds = [7, 21, 42]
output = root / "outputs"
output.mkdir(exist_ok=True)
train, test = load_data(root / "data")
results, rule_results = [], []

for seed in dict.fromkeys(seeds):
    training, validation = split_data(train, seed)
    train_text, valid_text = make_text(training), make_text(validation)
    for name, model in build_models().items():
        model.fit(train_text, training["rule_violation"])
        probabilities = model.predict_proba(valid_text)[:, 1]
        metrics, per_rule = evaluate_model(validation, probabilities)
        results.append({"model": name, "seed": seed,
                        "validation_rows": len(validation), **metrics})
        rule_results.append(per_rule.assign(model=name, seed=seed))

results = pd.DataFrame(results)
results.to_csv(output / "results.csv", index=False)
pd.concat(rule_results, ignore_index=True).to_csv(output / "per_rule.csv", index=False)
summary = results.groupby("model")[["roc_auc", "mean_rule_auc", "accuracy", "f1"]].mean()
print_table("Ortalama doğrulama sonuçları", summary.reset_index())




Ortalama doğrulama sonuçları
+---------------------------------------------------------+
|          model  roc_auc  mean_rule_auc  accuracy     f1 |
+---------------------------------------------------------+
|  char_logistic   0.8140         0.8006    0.7082 0.7137 |
|          dummy   0.5000         0.5000    0.4894 0.6570 |
| tfidf_logistic   0.8068         0.7948    0.7173 0.7254 |
+---------------------------------------------------------+


## 7. Test tahminlerini oluştur

Ortalama yerel kural AUC değerine göre seçilen metin modeli tüm eğitim verisiyle tekrar eğitilir.
Bu doğrulama yeni kurallara genellemeyi ölçmez. Mevcut test dosyası 10 örnek içerir; test etiketleri olmadan başarı hesaplanamaz.


In [ ]:
best_name = summary.drop(index="dummy")["mean_rule_auc"].idxmax()
final_model = build_models()[best_name]
final_model.fit(make_text(train), train["rule_violation"])
submission = pd.DataFrame({
    "row_id": test["row_id"],
    "rule_violation": final_model.predict_proba(make_text(test))[:, 1],
})
sample = pd.read_csv(root / "data" / "sample_submission.csv")
if not submission["row_id"].equals(sample["row_id"]):
    raise ValueError("Test ve sample_submission satır sıraları eşleşmiyor.")
submission.to_csv(output / "submission.csv", index=False)
print(f"Eğitim: {len(train)} satır, test: {len(test)} satır")
print(f"Seçilen model: {best_name}. Sonuçlar: {output}")


Eğitim: 2029 satır, test: 10 satır
Seçilen model: char_logistic. Sonuçlar: /content/outputs


## 8. Sonuçları indir
ZIP içinde results.csv, per_rule.csv ve submission.csv bulunur. Kaggle gönderimi yapılmaz.


In [ ]:
import shutil

result_zip = shutil.make_archive(str(root / "jigsaw_outputs"), "zip", root / "outputs")
files.download(result_zip)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>